In [1]:
import pandas as pd

<h2>Data Loading</h2>

Here, I loaded the 3 raw datasets that we will be using. I has also loaded the lookup tables, though they are not visible yet and will be made clearer in the data cleaning process. Below I have displayed the top 5 rows of each of the raw datasets. 

In [2]:
#Importing music micro dataset, without id columns value pairs.
file_path = 'raw_data/musicmicro/listening_data.txt'

music_micro_raw = pd.read_csv(
    file_path,
    sep="\t",
    header=0,
    index_col=False)

music_micro_raw.head()

,twitter-id,user-id,month,weekday,longitude,latitude,country-id,city-id,artist-id,track-id
0,134243699369590784,74717431,11,2,-50.531223,-18.453224,0,0,450514,7748381
1,134243700380401664,127821914,11,2,106.834450,-6.232960,1,1,202085,3529910
2,134243869201154048,174194590,11,2,-0.142822,51.520740,2,2,330061,5762915
3,134244034020524032,141847381,11,2,106.752390,-6.576483,1,3,404350,6987845
4,134244371557122048,87215499,11,2,-73.989890,40.738976,3,4,227460,4082536


In [3]:
#Importing raw mental health dataset.
mental_health_data_raw = pd.read_csv('raw_data/Mental Health Dataset.csv')
mental_health_data_raw.head()

,Timestamp,Gender,Country,Occupation,self_employed,family_history,treatment,Days_Indoors,Growing_Stress,Changes_Habits,Mental_Health_History,Mood_Swings,Coping_Struggles,Work_Interest,Social_Weakness,mental_health_interview,care_options
0,2014-08-27 11:29:31,Female,United States,Corporate,NaN,No,Yes,1-14 days,Yes,No,Yes,Medium,No,No,Yes,No,Not sure
1,2014-08-27 11:31:50,Female,United States,Corporate,NaN,Yes,Yes,1-14 days,Yes,No,Yes,Medium,No,No,Yes,No,No
2,2014-08-27 11:32:39,Female,United States,Corporate,NaN,Yes,Yes,1-14 days,Yes,No,Yes,Medium,No,No,Yes,No,Yes
3,2014-08-27 11:37:59,Female,United States,Corporate,No,Yes,Yes,1-14 days,Yes,No,Yes,Medium,No,No,Yes,Maybe,Yes
4,2014-08-27 11:43:36,Female,United States,Corporate,No,Yes,Yes,1-14 days,Yes,No,Yes,Medium,No,No,Yes,No,Yes


In [4]:

# load lines
with open("raw_data/songNames.txt", "r", encoding="utf-8") as f:
    cal_500_raw = pd.DataFrame({"artist_track": [line.strip() for line in f if line.strip()]})

# split artist + track
cal_500_raw[["artist", "track"]] = cal_500_raw["artist_track"].str.split("-", n=1, expand=True)

# clean text
def clean(s):
    return (
        s.str.lower()
         .str.replace("_", "", regex=False)
         .str.replace(r"[^\w\s]", "", regex=True)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )

cal_500_raw["artist"] = clean(cal_500_raw["artist"])
cal_500_raw["track"] = clean(cal_500_raw["track"])

# optional: drop original column
cal_500_raw = cal_500_raw.drop(columns="artist_track")

cal_500_raw.head()

,artist,track
0,10cc,foryouandi
1,2pac,trapped
2,5thdimension,onelessbelltoanswer
3,atribecalledquest,bonitaapplebum
4,aaronneville,tellitlikeitis


In [5]:
#Added vectorized tags to the cal_500 dataset.
hard = pd.read_csv("raw_data/hardAnnotations.txt", header=None)  # add the full path if needed
hard.columns = pd.read_csv("raw_data/vocab.txt", header=None)[0].tolist()
music_tagging = pd.concat([cal_500_raw, hard], axis=1)
music_tagging.head()

,artist,track,Emotion-Angry_/_Agressive,NOT-Emotion-Angry_/_Agressive,Emotion-Arousing_/_Awakening,NOT-Emotion-Arousing_/_Awakening,Emotion-Bizarre_/_Weird,NOT-Emotion-Bizarre_/_Weird,Emotion-Calming_/_Soothing,NOT-Emotion-Calming_/_Soothing,...,Genre-Best-World,Instrument_-_Acoustic_Guitar-Solo,Instrument_-_Electric_Guitar_(clean)-Solo,Instrument_-_Electric_Guitar_(distorted)-Solo,Instrument_-_Female_Lead_Vocals-Solo,Instrument_-_Harmonica-Solo,Instrument_-_Male_Lead_Vocals-Solo,Instrument_-_Piano-Solo,Instrument_-_Saxophone-Solo,Instrument_-_Trumpet-Solo
0,10cc,foryouandi,0,1,0,1,0,1,1,0,...,0,0,0,0,0,0,1,0,0,0
1,2pac,trapped,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
2,5thdimension,onelessbelltoanswer,0,0,0,0,0,1,1,0,...,0,1,0,0,0,0,0,0,0,0
3,atribecalledquest,bonitaapplebum,0,1,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,aaronneville,tellitlikeitis,0,1,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0


<h2>Data Cleaning</h2>

Adding track and artist names to music micro:

In [6]:
track_mapping = pd.read_csv('raw_data/musicmicro/track_mapping.txt', sep="\t")
artist_mapping = pd.read_csv('raw_data/musicmicro/artist_mapping.txt', sep="\t")

artist_mapping['artist'] = artist_mapping['artist'].str.lower().str.replace(' ', '')
track_mapping['track'] = track_mapping['track'].str.lower().str.replace(' ', '')


In [7]:
g = pd.merge(music_micro_raw, artist_mapping, on='artist-id', how='inner')
music_micro_ta = pd.merge(g, track_mapping, on='track-id', how='inner')
music_micro_ta

,twitter-id,user-id,month,weekday,longitude,latitude,country-id,city-id,artist-id,track-id,artist,track
0,134243699369590784,74717431,11,2,-50.531223,-18.453224,0,0,450514,7748381,tihuana,naparededoquintal
1,134243700380401664,127821914,11,2,106.834450,-6.232960,1,1,202085,3529910,jamesmorrison,yougivemesomething
2,134243869201154048,174194590,11,2,-0.142822,51.520740,2,2,330061,5762915,petshopboys,westendgirls
3,134244034020524032,141847381,11,2,106.752390,-6.576483,1,3,404350,6987845,suede,everythingwillflow
4,134244371557122048,87215499,11,2,-73.989890,40.738976,3,4,227460,4082536,kaskade,ice
...,...,...,...,...,...,...,...,...,...,...,...,...
452751,250847226383978496,70563820,9,2,110.340640,-7.802276,1,1907,301871,9530975,natashabedingfield,betweentheraindrops
452752,250847725116071936,370457976,9,2,107.626190,-6.905580,1,114,215116,3848019,johnmayer,somethinglikeolivia
452753,250849073052143616,134108604,9,2,-76.718850,39.955963,3,548,283679,5020296,michaelfranti&spearhead,sayhey
452754,250849250706092032,219099036,9,2,-111.669560,40.220425,3,5091,68747,1224442,carteldesanta,traficandorimas


Merge and create final Music dataset 

In [8]:
matched = pd.merge(music_micro_ta, music_tagging, on=['artist', 'track'], how='inner')
matched

,twitter-id,user-id,month,weekday,longitude,latitude,country-id,city-id,artist-id,track-id,...,Genre-Best-World,Instrument_-_Acoustic_Guitar-Solo,Instrument_-_Electric_Guitar_(clean)-Solo,Instrument_-_Electric_Guitar_(distorted)-Solo,Instrument_-_Female_Lead_Vocals-Solo,Instrument_-_Harmonica-Solo,Instrument_-_Male_Lead_Vocals-Solo,Instrument_-_Piano-Solo,Instrument_-_Saxophone-Solo,Instrument_-_Trumpet-Solo
0,134266061230047232,314649657,11,2,-51.177275,-29.915337,0,86,343878,5986620,...,0,0,0,0,0,0,0,0,0,0
1,134312266639212544,329400087,11,2,-45.917521,-23.061418,0,306,78492,1403960,...,0,0,0,0,0,0,0,0,0,1
2,134339142418042880,105875448,11,2,-43.447310,-22.916032,0,50,393462,6800302,...,0,0,0,0,0,0,0,0,0,0
3,134350511678832641,108600330,11,2,-76.639496,39.304510,3,297,414087,7130125,...,0,0,0,0,0,0,0,0,0,0
4,134357350260809730,39820216,11,2,4.904054,52.354750,14,98,393462,6800302,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2692,250395709398986752,26432623,9,1,-122.912750,49.199432,5,387,468091,8029759,...,0,1,0,0,0,0,0,0,0,0
2693,250533563949277185,325338295,9,1,106.826795,-6.168961,1,17,477249,8182052,...,0,0,1,1,0,0,0,0,0,0
2694,250627160333770754,49146752,9,1,-46.595545,-23.682814,0,5,51218,902128,...,0,0,0,0,0,0,0,0,0,0
2695,250637024867008512,423348268,9,1,35.243322,38.963745,9,33,66022,1172796,...,0,0,0,0,0,0,0,0,0,0
